In [22]:
import lyricsgenius
import json
from time import sleep
import re
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
import ssl
import nltk
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
from PIL import Image
from wordcloud import WordCloud, ImageColorGenerator
from nltk.probability import FreqDist

In [9]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Скачиваем все необходимые ресурсы NLTK
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


# Необходимые ресурсы NLTK 
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

Я буду использовать API Genius для автоматического сбора текстов песен Billie Eilish и Adele из их альбомов и синглов, перечисленных в словаре `billie_releases` и `adele_releases`. Код подключится к Genius, найдёт каждую песню, обработает особые случаи (например, коллаборации с другими артистами), а затем сохранит все тексты в структурированный JSON-файл `billie_eilish.json` и `adele.json`. В конце получится два готовых файла с текстами всех обработанных треков.

In [12]:
def get_artist_lyrics(artist_name, releases, output_file):
   
    """
    Функция для получения текстов песен конкретного исполнителя.
    Создает клиент Genius API, обрабатывает все указанные релизы,
    ищет тексты песен с обработкой особых случаев (коллаборации),
    сохраняет результаты в JSON-файл.
    """

    GENIUS_API_TOKEN = "Sat2ONjCO2-EiyYzgziivZlQY9MFFuOQXwOKK5qpJ3ahsiuORYTov_qw-HpwOaJJ"  # Токен для доступа к Genius API
    
    genius = lyricsgenius.Genius(GENIUS_API_TOKEN)
    genius.verbose = False
    genius.skip_non_songs = True  # Пропускаем не-песни
    genius.excluded_terms = ["(Remix)", "(Live)", "(Version)"]  # Исключаем специфичные версии
    genius.timeout = 30
    all_lyrics = {}  # Словарь для хранения всех текстов
    
    # Обрабатываем каждый релиз и его треки
    for release, tracks in releases.items():
        print(f"\nОбрабатываю: {release}")
        release_lyrics = {}
        
        for track in tracks:
            try:
                # Особые случаи для коллабораций
                if track.lower() == "guess":
                    song = genius.search_song(track, "Charli XCX")
                    if not song:
                        song = genius.search_song(track, artist_name)
                elif track.lower() == "lovely":
                    song = genius.search_song(track, "Khalid")
                    if not song:
                        song = genius.search_song(track, artist_name)
                elif track.lower() == "lo vas a olvidar":
                    song = genius.search_song(track, "ROSALÍA")
                    if not song:
                        song = genius.search_song(track, artist_name)
                else:
                    song = genius.search_song(track, artist_name)
                
                if song:
                    print(f"  Найдена: {song.title}")
                    release_lyrics[song.title] = song.lyrics
                    sleep(2)  # Задержка между запросами
                else:
                    print(f"  Не найдена: {track}")
            except Exception as e:
                print(f"  Ошибка при поиске {track}: {str(e)}")
        
        if release_lyrics:
            all_lyrics[release] = release_lyrics
        else:
            print(f"  Не удалось получить песни для {release}")

    # Сохраняем в JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_lyrics, f, ensure_ascii=False, indent=2)

    print(f"\nРезультаты сохранены в {output_file}")

def main():
    # Данные для Billie Eilish
    billie_releases = {
        "HIT ME HARD AND SOFT": [
            "SKINNY", "LUNCH", "CHIHIRO", "BIRDS OF A FEATHER",
            "WILDFLOWER", "THE GREATEST", "L'AMOUR DE MA VIE",
            "THE DINER", "BITTERSUITE", "BLUE"
        ],
        "Happier Than Ever": [
            "Getting Older", "I Didn't Change My Number", "Billie Bossa Nova",
            "my future", "Oxytocin", "GOLDWING", "Lost Cause",
            "Halley's Comet", "Not My Responsibility", "OverHeated",
            "Everybody Dies", "Your Power", "NDA", "Therefore I Am",
            "Happier Than Ever", "Male Fantasy"
        ],
        "WHEN WE ALL FALL ASLEEP, WHERE DO WE GO?": [
            "bad guy", "xanny", "you should see me in a crown",
            "all the good girls go to hell", "wish you were gay",
            "when the party's over", "8", "my strange addiction",
            "bury a friend", "ilomilo", "listen before i go", "i love you",
            "goodbye"
        ],
        "Don't Smile at Me": [
            "COPYCAT", "idontwannabeyouanymore", "my boy", "watch",
            "party favor", "bellyache", "ocean eyes", "hostage"
        ],
        "Guitar Songs": [
            "TV", "The 30th"
        ],
        "everything i wanted": ["everything i wanted"],
        "lovely": ["lovely"],  # с Khalid
        "No Time To Die": ["No Time To Die"],
        "bored": ["bored"],
        "Six Feet Under": ["Six Feet Under"],
        "Lo Vas A Olvidar": ["Lo Vas A Olvidar"],  # с ROSALÍA
        "Bitches Broken Hearts": ["Bitches Broken Hearts"],
        "Come Out and Play": ["Come Out and Play"],
        "When I Was Older": ["When I Was Older"],
        "Guess": ["Guess featuring billie eilish"],  # с Charli XCX
        "What Was I Made For?": ["What Was I Made For?"],
        "hotline (edit)": ["hotline (edit)"]
    }

    # Данные для Adele
    adele_releases = {
        "30": [
            "Strangers By Nature", "Easy On Me", "My Little Love",
            "Cry Your Heart Out", "Oh My God", "Can I Get It",
            "I Drink Wine", "All Night Parking Interlude", 
            "Woman Like Me", "Hold On", "To Be Loved",
            "Love Is A Game"
        ],
        "25": [
            "Hello", "Send My Love (To Your New Lover)", "I Miss You",
            "When We Were Young", "Remedy", "Water Under the Bridge",
            "River Lea", "Love in the Dark", "Million Years Ago",
            "All I Ask", "Sweetest Devotion"
        ],
        "21": [
            "Rolling in the Deep", "Rumour Has It", "Turning Tables",
            "Don't You Remember", "Set Fire to the Rain",
            "He Won't Go", "Take It All", "I'll Be Waiting",
            "One and Only", "Lovesong", "Someone Like You",
            "I Found a Boy"
        ],
        "19": [
            "Daydreamer", "Best for Last", "Chasing Pavements", 
            "Cold Shoulder", "Crazy for You", "Melt My Heart to Stone",
            "First Love", "Right as Rain", "Make You Feel My Love",
            "My Same", "Tired", "Hometown Glory"
        ],
        "Skyfall": ["Skyfall"]
    }

    # Получаем тексты для обоих исполнителей
    get_artist_lyrics("Billie Eilish", billie_releases, "billie_eilish.json")
    get_artist_lyrics("Adele", adele_releases, "adele.json")

if __name__ == "__main__":
    main()


Обрабатываю: HIT ME HARD AND SOFT
  Найдена: SKINNY
  Найдена: LUNCH
  Найдена: CHIHIRO
  Найдена: BIRDS OF A FEATHER
  Найдена: WILDFLOWER
  Найдена: THE GREATEST
  Найдена: L’AMOUR DE MA VIE
  Найдена: THE DINER
  Найдена: BITTERSUITE
  Найдена: BLUE

Обрабатываю: Happier Than Ever
  Найдена: Getting Older
  Найдена: I Didn’t Change My Number
  Найдена: Billie Bossa Nova
  Найдена: my future
  Найдена: Oxytocin
  Найдена: GOLDWING
  Найдена: Lost Cause
  Найдена: Halley’s Comet
  Найдена: Not My Responsibility
  Найдена: OverHeated
  Найдена: Everybody Dies
  Найдена: Your Power
  Найдена: NDA
  Найдена: Therefore I Am
  Найдена: Happier Than Ever
  Найдена: Male Fantasy

Обрабатываю: WHEN WE ALL FALL ASLEEP, WHERE DO WE GO?
  Найдена: bad guy
  Найдена: xanny
  Найдена: you should see me in a crown
  Найдена: all the good girls go to hell
  Найдена: wish you were gay
  Найдена: when the party’s over
  Найдена: 8
  Найдена: my strange addiction
  Найдена: bury a friend
  Найдена: il

Я буду очищать тексты песен Billie Eilish и Adele из уже собранных JSON-файлов (`billie_eilish.json` и `adele.json`). Код выполнит следующие действия:
1. Для каждого исполнителя загрузит исходные тексты песен из JSON-файла
2. Очистит каждый текст, удаляя:
   - Всё до фразы "Read More" включительно
   - Текст в квадратных скобках (например, [Verse 1])
   - Лишние пробелы и пустые строки
3. Для каждой песни выведет информацию об очистке в формате: "Очищено: [Альбом] - [Песня] ([количество символов])"
4. Сохранит очищенные тексты в новые JSON-файлы:
   - `billie_eilish_cleaned.json` для Billie Eilish
   - `adele_cleaned.json` для Adele
5. После обработки каждого файла выведет сообщение о сохранении с указанием имени файла

В результате получатся два новых файла с чистыми текстами песен, где:
- Сохранена оригинальная структура (альбомы → песни)
- Тексты приведены к единому формату без лишних элементов
- Все данные корректно форматированы с отступами в JSON-файлах

Код обрабатывает файлы независимо друг от друга, сохраняя раздельные результаты для каждого исполнителя.

In [13]:
def clean_lyrics(raw_text):
    
    """
    Очищает текст песни от лишних элементов.
    Параметры:
        Исходный текст песни с возможным мусором
    Возвращает:
        Очищенный текст или исходный объект, если передан не текст
    """
    
    if not isinstance(raw_text, str):
        return raw_text
        
    # Удаляем всё до "Read More" включительно
    cleaned = re.sub(r'^.*?Read More\s*', '', raw_text, flags=re.DOTALL | re.IGNORECASE)
    
    # Удаляем все квадратные скобки и их содержимое
    cleaned = re.sub(r'\[.*?\]', '', cleaned)
    
    # Удаляем лишние пробелы и пустые строки
    cleaned = '\n'.join([line.strip() for line in cleaned.split('\n') if line.strip()])
    
    return cleaned.strip()

def process_artist_file(input_file, output_file):
   
    """
    Обрабатывает файл с текстами песен одного исполнителя
    Параметры:
        input_file: Путь к исходному JSON-файлу
        output_file: Путь для сохранения очищенного JSON-файла
    """

    # Загружаем исходные тексты
    with open(input_file, 'r', encoding='utf-8') as f:
        songs_data = json.load(f)
    
    # Очищаем тексты
    cleaned_songs = {}
    for album_name, tracks in songs_data.items():
        cleaned_songs[album_name] = {}
        for song_name, lyrics in tracks.items():
            cleaned = clean_lyrics(lyrics)
            cleaned_songs[album_name][song_name] = cleaned
            print(f"Очищено: {album_name} - {song_name} ({len(cleaned)} символов)")
    
    # Сохраняем очищенные тексты
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_songs, f, ensure_ascii=False, indent=2)
    
    print(f"\nОчищенные тексты сохранены в {output_file}\n")
    

# Обрабатываем оба файла отдельно
process_artist_file('billie_eilish.json', 'billie_eilish_cleaned.json')
process_artist_file('adele.json', 'adele_cleaned.json')

Очищено: HIT ME HARD AND SOFT - SKINNY (759 символов)
Очищено: HIT ME HARD AND SOFT - LUNCH (1498 символов)
Очищено: HIT ME HARD AND SOFT - CHIHIRO (1806 символов)
Очищено: HIT ME HARD AND SOFT - BIRDS OF A FEATHER (1428 символов)
Очищено: HIT ME HARD AND SOFT - WILDFLOWER (1482 символов)
Очищено: HIT ME HARD AND SOFT - THE GREATEST (1188 символов)
Очищено: HIT ME HARD AND SOFT - L’AMOUR DE MA VIE (1852 символов)
Очищено: HIT ME HARD AND SOFT - THE DINER (1623 символов)
Очищено: HIT ME HARD AND SOFT - BITTERSUITE (1592 символов)
Очищено: HIT ME HARD AND SOFT - BLUE (1918 символов)
Очищено: Happier Than Ever - Getting Older (1540 символов)
Очищено: Happier Than Ever - I Didn’t Change My Number (813 символов)
Очищено: Happier Than Ever - Billie Bossa Nova (1493 символов)
Очищено: Happier Than Ever - my future (893 символов)
Очищено: Happier Than Ever - Oxytocin (1420 символов)
Очищено: Happier Than Ever - GOLDWING (894 символов)
Очищено: Happier Than Ever - Lost Cause (1775 символов)
Очи

Я буду загружать и структурировать данные из JSON-файлов с текстами песен Billie Eilish и Adele, преобразовывая их в удобные DataFrame с разделением на альбомы и синглы. Код создаст отдельные CSV-файлы для каждого исполнителя и объединённый файл, добавив информацию о количестве слов и символов в каждом тексте, а также корректно обработает возможные ошибки при загрузке данных. В результате будут созданы три CSV-файла с чёткой структурой данных для последующего анализа.

In [14]:
def load_and_create_dataframe(filepath, artist_name):
   
    """
    Создает DataFrame из JSON файла с четким разделением на альбомы и синглы
    Параметры:
        filepath: Путь к JSON файлу
        artist_name: Имя исполнителя
    Возвращает:
        pd.DataFrame с колонками:
        - artist: Исполнитель
        - song: Название песни
        - album_name: Название альбома или 'single' для синглов
        - lyrics: Текст песни
        - word_count: Количество слов
        - char_count: Количество символов
    """
   
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    result = []
    
    for key, value in data.items():
        # Если value - словарь, то это альбом с треками
        if isinstance(value, dict):
            for song_name, lyrics in value.items():
                result.append({
                    'artist': artist_name,
                    'song': song_name,
                    'album_name': key,  # Название альбома
                    'lyrics': lyrics,
                    'word_count': len(lyrics.split()) if lyrics else 0,
                    'char_count': len(lyrics) if lyrics else 0
                })
        # Если value - список, то это сингл
        elif isinstance(value, list):
            lyrics = value[0] if value else ''
            result.append({
                'artist': artist_name,
                'song': key,  # Название сингла
                'album_name': 'single',  # Явно указываем что это сингл
                'lyrics': lyrics,
                'word_count': len(lyrics.split()) if lyrics else 0,
                'char_count': len(lyrics) if lyrics else 0
            })
    
    return pd.DataFrame(result)

# Обработка файлов с проверкой ошибок
try:
    # Загрузка данных
    billie_df = load_and_create_dataframe('billie_eilish_cleaned.json', 'Billie Eilish')
    adele_df = load_and_create_dataframe('adele_cleaned.json', 'Adele')
    
    # Объединение данных
    combined_df = pd.concat([billie_df, adele_df], ignore_index=True)
    
    # Сохранение в CSV
    billie_df.to_csv('billie_eilish_songs.csv', index=False, encoding='utf-8')
    adele_df.to_csv('adele_songs.csv', index=False, encoding='utf-8')
    combined_df.to_csv('all_songs.csv', index=False, encoding='utf-8')
    
    # Проверка результатов
    print("Файлы созданы:")
    print(f"1. billie_songs.csv ({len(billie_df)} записей)")
    print(f"2. adele_songs.csv ({len(adele_df)} записей)")
    print(f"3. all_songs.csv ({len(combined_df)} записей)")

except FileNotFoundError as e:
    print(f"Ошибка: файл не найден - {e.filename}")
except json.JSONDecodeError:
    print("Ошибка: некорректный формат JSON файла")
except Exception as e:
    print(f"Неожиданная ошибка: {str(e)}")

Файлы созданы:
1. billie_songs.csv (61 записей)
2. adele_songs.csv (48 записей)
3. all_songs.csv (109 записей)


Следующий код будет выполнять автоматизированный разведывательный анализ (EDA) текстов песен двух исполнителей. 

1. Код генерирует три типа графиков для каждого исполнителя - распределение количества слов, распределение символов и количество песен по альбомам, сохраняя их в папку `visualizations`. Каждый график содержит ключевые статистики (среднее, медиану, экстремумы) в информационных блоках.

2. Сравнительный анализ: Функция `compare_artists` создает сводную таблицу с метриками обоих исполнителей и строит сравнительные графики по средним значениям слов, символов и количеству песен, выделяя различия цветами (синий для Billie Eilish, оранжевый для Adele).

3. Обработка данных: Перед визуализацией код выводит основную информацию о датасетах - структуру данных, описательные статистики, количество пропусков и уникальные альбомы, что помогает оценить качество данных перед анализом.

In [15]:
# Создаем папку для графиков, если ее нет
os.makedirs('visualizations', exist_ok=True) 

def save_plot(fig, filename):

    """Сохраняет график в папку visualizations"""
    
    path = os.path.join('visualizations', filename)
    fig.savefig(path, bbox_inches='tight', dpi=300)
    plt.close()
    
def perform_eda(df, artist_name):
    
    """
    Выполняет разведывательный анализ данных с улучшенной визуализацией
    """
    
    print(f"\n\n Анализ для {artist_name} \n")
    
    # Основная информация
    print("1. Общая информация:")
    print(df.info())
    
    print("\n2. Описательные статистики:")
    print(df.describe())
    
    print("\n3. Количество пропусков:")
    print(df.isnull().sum())
    
    print("\n4. Уникальные альбомы:")
    print(df['album_name'].unique())
    
    print("\n5. Первые 5 записей:")
    print(df.head())

    # Улучшенная палитра цветов
    if artist_name == "Billie Eilish":
        palette = sns.color_palette("husl", len(df['album_name'].unique()))
    else:
        palette = sns.color_palette("husl", len(df['album_name'].unique()))

    # 1. Распределение количества слов
    plt.figure(figsize=(12, 6))
    ax = sns.histplot(df['word_count'], bins=20, kde=True, color='#4e79a7')
    
    # Добавляем статистики в квадратик
    mean_val = df['word_count'].mean()
    median_val = df['word_count'].median()
    
    stats_text = f"Среднее (mean): {mean_val:.1f}\nМедиана (median): {median_val:.1f}\n\nОбщая статистика:\nКоличество песен: {len(df)}\nМакс. слов: {df['word_count'].max()}\nМин. слов: {df['word_count'].min()}"
    bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="#4e79a7", lw=1.5, alpha=0.9)
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes,
            ha='right', va='top', bbox=bbox_props, fontsize=10)
    
    # Добавляем линии среднего и медианы с легендой
    ax.axvline(mean_val, color='#e15759', linestyle='--', linewidth=1.5, alpha=0.7, label='Среднее (mean)')
    ax.axvline(median_val, color='#76b7b2', linestyle='-', linewidth=1.5, alpha=0.7, label='Медиана (median)')
    
    # Добавляем легенду
    ax.legend(loc='upper left', bbox_to_anchor=(0.05, 0.95), frameon=True, framealpha=0.9)
    
    plt.title(f'{artist_name}: Распределение количества слов', pad=15)
    save_plot(plt.gcf(), f'{artist_name.lower()}_word_dist.png')

    # 2. Распределение количества символов
    plt.figure(figsize=(12, 6))
    ax = sns.histplot(df['char_count'], bins=20, kde=True, color='#f28e2b')
    
    mean_val = df['char_count'].mean()
    median_val = df['char_count'].median()
    
    stats_text = f"Среднее (mean): {mean_val:.1f}\nМедиана (median): {median_val:.1f}\n\nОбщая статистика:\nКоличество песен: {len(df)}\nМакс. символов: {df['char_count'].max()}\nМин. символов: {df['char_count'].min()}"
    bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="#f28e2b", lw=1.5, alpha=0.9)
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes,
            ha='right', va='top', bbox=bbox_props, fontsize=10)
    
    # Добавляем линии среднего и медианы с легендой
    ax.axvline(mean_val, color='#e15759', linestyle='--', linewidth=1.5, alpha=0.7, label='Среднее (mean)')
    ax.axvline(median_val, color='#76b7b2', linestyle='-', linewidth=1.5, alpha=0.7, label='Медиана (median)')
    
    # Добавляем легенду
    ax.legend(loc='upper left', bbox_to_anchor=(0.05, 0.95), frameon=True, framealpha=0.9)
    
    plt.title(f'{artist_name}: Распределение количества символов', pad=15)
    save_plot(plt.gcf(), f'{artist_name.lower()}_char_dist.png')

    # 3. Количество песен по альбомам
    plt.figure(figsize=(12, 6))
    album_counts = df['album_name'].value_counts()
    bar = sns.barplot(x=album_counts.values, y=album_counts.index, 
                     hue=album_counts.index, palette=palette, legend=False, dodge=False)
    
    for i, v in enumerate(album_counts.values):
        bar.text(v + 0.2, i, str(v), color='black', ha='left', va='center', fontsize=9)
    
    # Добавляем общую статистику по альбомам
    stats_text = f"Всего альбомов: {len(album_counts)}\nВсего песен: {len(df)}"
    bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="#59a14f", lw=1.5, alpha=0.9)
    plt.text(0.95, 0.95, stats_text, transform=plt.gca().transAxes,
            ha='right', va='top', bbox=bbox_props, fontsize=10)
    
    plt.title(f'{artist_name}: Количество песен по альбомам', pad=15)
    save_plot(plt.gcf(), f'{artist_name.lower()}_album_counts.png')
    
def compare_artists(billie_df, adele_df):
    """Сравнивает статистику двух исполнителей"""
    print("\n\n Сравнение исполнителей")
    
    combined = pd.concat([billie_df.assign(artist='Billie Eilish'), 
                         adele_df.assign(artist='Adele')])
    
    stats = combined.groupby('artist').agg({
        'word_count': ['mean', 'median', 'max', 'min'],
        'char_count': ['mean', 'median', 'max', 'min'],
        'song': 'count',
        'album_name': 'nunique'
    }).reset_index()
    
    stats.columns = ['artist', 'mean_words', 'median_words', 'max_words', 'min_words',
                    'mean_chars', 'median_chars', 'max_chars', 'min_chars', 
                    'song_count', 'album_count']
    
    print("\nСравнительная статистика:")
    print(stats)
    
    # Цвета для исполнителей
    colors = ['#4e79a7', '#f28e2b']
    
    plt.figure(figsize=(16, 8))
    
    # 1. Среднее количество слов
    plt.subplot(2, 3, 1)
    bar1 = sns.barplot(data=stats, x='artist', y='mean_words', 
                      hue='artist', palette=colors, legend=False)
    plt.title('Среднее количество слов', pad=15)
    plt.xlabel('')
    for p in bar1.patches:
        bar1.annotate(f"{p.get_height():.1f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), 
                     textcoords='offset points', fontsize=10)
    
    # 2. Среднее количество символов
    plt.subplot(2, 3, 2)
    bar2 = sns.barplot(data=stats, x='artist', y='mean_chars', 
                      hue='artist', palette=colors, legend=False)
    plt.title('Среднее количество символов', pad=15)
    plt.xlabel('')
    for p in bar2.patches:
        bar2.annotate(f"{p.get_height():.1f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), 
                     textcoords='offset points', fontsize=10)
    
    # 3. Количество песен
    plt.subplot(2, 3, 3)
    bar3 = sns.barplot(data=stats, x='artist', y='song_count', 
                      hue='artist', palette=colors, legend=False)
    plt.title('Количество песен', pad=15)
    plt.xlabel('')
    for p in bar3.patches:
        bar3.annotate(f"{int(p.get_height())}", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), 
                     textcoords='offset points', fontsize=10)
    
    plt.suptitle('Сравнение Billie Eilish и Adele', y=1.02, fontsize=14)
    plt.tight_layout()
    save_plot(plt.gcf(), 'artists_comparison.png')

# Загрузка данных
try:
    billie_df = pd.read_csv('billie_eilish_songs.csv')
    adele_df = pd.read_csv('adele_songs.csv')
    
    # Выполняем EDA для каждого исполнителя
    perform_eda(billie_df, "Billie Eilish")
    perform_eda(adele_df, "Adele")
    
    # Сравниваем исполнителей
    compare_artists(billie_df, adele_df)
    
    print("\nАнализ завершен. Графики сохранены в папку 'visualizations'")
    
except FileNotFoundError as e:
    print(f"Ошибка: файл не найден - {e.filename}")
except Exception as e:
    print(f"Неожиданная ошибка: {str(e)}")



 Анализ для Billie Eilish 

1. Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   artist      61 non-null     object
 1   song        61 non-null     object
 2   album_name  61 non-null     object
 3   lyrics      61 non-null     object
 4   word_count  61 non-null     int64 
 5   char_count  61 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 3.0+ KB
None

2. Описательные статистики:
       word_count   char_count
count   61.000000    61.000000
mean   243.295082  1205.278689
std     76.148396   375.443344
min     37.000000   189.000000
25%    187.000000   903.000000
50%    242.000000  1188.000000
75%    287.000000  1482.000000
max    407.000000  1918.000000

3. Количество пропусков:
artist        0
song          0
album_name    0
lyrics        0
word_count    0
char_count    0
dtype: int64

4. Уникальные альбомы:
['HIT

**Выводы по анализу текстов песен Adele и Billie Eilish**  
1. Длина текстов и стилистика  
   - Adele демонстрирует более длинные и содержательные тексты: средние значения количества слов (318 vs 243) и символов (1553 vs 1205) значительно выше, чем у Billie Eilish. 
2. Консистентность vs вариативность 
   - У Adele распределение длины песен более однородное (симметричные графики, близкие значения среднего и медианы), что говорит о предсказуемости структуры ее песен.  
   - У Billie Eilish заметен больший разброс: от очень коротких (37 слов, 189 символов) до длинных композиций (407 слов, 1918 символов). Это отражает экспериментальный подход и разнообразие форматов в ее творчестве.  
3. Количество песен и продуктивность**
   - В выборке Billie Eilish представлено больше песен (61 vs 48), но их тексты в среднем короче. 

Этот код выполняет комплексный анализ текстов песен Billie Eilish и Adele, создавая визуализации в виде облаков слов. Сначала обрабатывает тексты песен: удаляет стоп-слова и пунктуацию, выполняет лемматизацию слов (приведение к начальной форме) с учетом разных частей речи, сохраняя при этом апострофы в словах. После обработки генерирует три типа облаков слов - отдельные для каждого исполнителя и сравнительное, где слова окрашены в разные цвета в зависимости от их принадлежности (синий для уникальных слов Billie Eilish, оранжевый для уникальных слов Adele и серый для общих слов). Облака создаются с учетом частоты употребления слов и сохраняются в папку wordclouds.

In [16]:
# Создаем папку для облаков слов
os.makedirs('wordclouds', exist_ok=True)

def process_text(text):
    # Кастомные стоп-слова
    custom_stopwords = {'oh', 'na', 'la', 'hey', 'like', 'im', 'dont', 'wont', 'cant', 
                       'know', 'just', 'want', 'get', 'got', 'go', 'll', 've', 're'}
    stop_words = set(stopwords.words('english')) | custom_stopwords
    
    # Удаляем пунктуацию, но сохраняем апострофы
    text = re.sub(r"[^\w\s']", ' ', str(text).lower())
    
    # Простая токенизация без использования Punkt
    words = []
    for token in text.split():
        # Разделяем слова с апострофами (например, "don't" -> ["don", "t"])
        parts = token.split("'")
        for part in parts:
            if part:  # Игнорируем пустые строки
                words.append(part)
    
    # Улучшенная лемматизация
    lemmatizer = WordNetLemmatizer()
    lemmatized = []
    for word in words:
        if word.isalpha() and word not in stop_words:
            # Пробуем разные части речи для лучшей лемматизации
            for pos in ['n', 'v', 'a', 'r']:
                lemma = lemmatizer.lemmatize(word, pos=pos)
                if lemma != word:  # Если лемматизация дала результат
                    word = lemma
                    break
            lemmatized.append(word)
    
    return ' '.join(lemmatized)

# Загрузка и обработка текстов
billie_text = ' '.join(billie_df['lyrics'].apply(process_text))
adele_text = ' '.join(adele_df['lyrics'].apply(process_text))

def generate_wordcloud(text, artist_name, filename, color_func=None):
    mask = np.array(Image.open("cloud_mask.png")) if os.path.exists("cloud_mask.png") else None
    
    # Создаем частотный словарь для устранения дубликатов
    word_freq = {}
    for word in text.split():
        word_freq[word] = word_freq.get(word, 0) + 1
    
    wc = WordCloud(
        width=1600, 
        height=800,
        background_color='white',
        max_words=200,
        mask=mask,
        contour_width=3,
        contour_color='steelblue',
        collocations=False  # Отключаем биграммы
    ).generate_from_frequencies(word_freq)
    
    if color_func:
        wc.recolor(color_func=color_func)
    
    plt.figure(figsize=(20, 10))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'{artist_name}: Облако слов', fontsize=24, pad=20)
    plt.axis('off')
    plt.savefig(f'wordclouds/{filename}', bbox_inches='tight', dpi=300)
    plt.close()

# Индивидуальные облака слов
generate_wordcloud(billie_text, 'Billie Eilish', 'billie_wordcloud.png')
generate_wordcloud(adele_text, 'Adele', 'adele_wordcloud.png')

def generate_comparison_cloud(text1, text2, name1, name2):
    # Создаем частотные распределения
    words1 = text1.split()
    words2 = text2.split()
    
    freq1 = FreqDist(words1)
    freq2 = FreqDist(words2)
    
    # Находим уникальные и общие слова (только частые)
    min_freq = 2  # Минимальная частота для включения в облако
    common_words = {w for w in set(words1) & set(words2) if freq1[w] > min_freq and freq2[w] > min_freq}
    unique1 = {w for w in set(words1) if freq1[w] > min_freq} - common_words
    unique2 = {w for w in set(words2) if freq2[w] > min_freq} - common_words
    
    # Функция для цветового кодирования
    def color_func(word, *args, **kwargs):
        if word in unique1:
            return 'rgb(78, 121, 167)'  # Синий для Billie
        elif word in unique2:
            return 'rgb(242, 142, 43)'   # Оранжевый для Adele
        else:
            return 'rgb(128, 128, 128)'  # Серый для общих
    
    # Создаем объединенный текст с весами
    word_weights = {}
    for word in unique1:
        word_weights[word] = freq1[word]
    for word in unique2:
        word_weights[word] = freq2[word]
    for word in common_words:
        word_weights[word] = (freq1[word] + freq2[word]) / 2
    
    # Генерируем облако
    mask = np.array(Image.open("cloud_mask.png")) if os.path.exists("cloud_mask.png") else None
    wc = WordCloud(
        width=1600, 
        height=800,
        background_color='white',
        max_words=200,
        mask=mask,
        contour_width=3,
        contour_color='steelblue',
        collocations=False
    ).generate_from_frequencies(word_weights)
    
    wc.recolor(color_func=color_func)
    
    plt.figure(figsize=(20, 10))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'{name1} vs {name2}: Сравнительное облако слов', fontsize=24, pad=20)
    plt.axis('off')
    plt.savefig('wordclouds/comparison_wordcloud.png', bbox_inches='tight', dpi=300)
    plt.close()

generate_comparison_cloud(billie_text, adele_text, 'Billie Eilish', 'Adele')

Код создаёт сравнительную диаграмму топ-20 слов каждого исполнителя.

In [24]:
# Функция для получения топ-20 слов
def get_top_words(text, n=40):
    words = text.split()
    return FreqDist(words).most_common(n)

# Получаем топ-20 слов для каждого исполнителя
billie_top = get_top_words(billie_text)
adele_top = get_top_words(adele_text)

# Визуализация
plt.figure(figsize=(18, 12))

# График для Billie Eilish
plt.subplot(1, 2, 1)
billie_words, billie_counts = zip(*billie_top)
sns.barplot(x=list(billie_counts), y=list(billie_words), palette='Blues_d')
plt.title('Billie Eilish: Топ-20 слов', fontsize=16, pad=20)
plt.xlabel('Частота', fontsize=12)
plt.ylabel('Слова', fontsize=12)

# График для Adele
plt.subplot(1, 2, 2)
adele_words, adele_counts = zip(*adele_top)
sns.barplot(x=list(adele_counts), y=list(adele_words), palette='Oranges_d')
plt.title('Adele: Топ-20 слов', fontsize=16, pad=20)
plt.xlabel('Частота', fontsize=12)
plt.ylabel('')

plt.tight_layout()
plt.savefig('wordclouds/top_words_comparison.png', bbox_inches='tight', dpi=300)
plt.close()

# Создаем DataFrame для общих и уникальных слов
common_words = set(billie_words) & set(adele_words)
billie_unique = set(billie_words) - common_words
adele_unique = set(adele_words) - common_words

print(f"Общие слова: {common_words}")
print(f"Уникальные слова Billie Eilish: {billie_unique}")
print(f"Уникальные слова Adele: {adele_unique}")

Общие слова: {'back', 'try', 'love', 'keep', 'leave', 'take', 'never', 'tell', 'look', 'one', 'ooh', 'time', 'good', 'think', 'say', 'let', 'cause', 'feel', 'see', 'make'}
Уникальные слова Billie Eilish: {'ah', 'way', 'hmm', 'eye', 'guess', 'wanna', 'come', 'friend', 'would', 'need', 'em', 'mind', 'call', 'really', 'might', 'bad', 'could', 'mm', 'da', 'maybe'}
Уникальные слова Adele: {'know', 'rumour', 'right', 'heart', 'yeah', 'always', 'world', 'every', 'gonna', 'thing', 'fall', 'u', 'young', 'cry', 'river', 'baby', 'give', 'light', 'fool', 'hold'}


**Выводы по анализу топ-20 песен Adele и Billie Eilish**  
1. Общие слова 
    - В текстах песен Billie Eilish и Adele встречаются 20 одинаковых слов (например, love, time, never), что указывает на схожие темы, такие как чувства, отношения и размышления.  
2. Уникальные слова Billie Eilish
    - В её текстах преобладают слова, передающие неуверенность (maybe, guess), эмоциональные междометия (hmm, ah), а также разговорные выражения (wanna, gonna), что отражает её стиль — более интимный и неформальный.  
3. Уникальные слова Adele
    - В её текстах чаще встречаются слова, связанные с сильными эмоциями (heart, cry), природными образами (river, light) и устойчивыми выражениями (right, always), что характерно для её мощного, драматического вокала и лирики. 

Этот код анализирует и сравнивает лексическое разнообразие текстов песен Billie Eilish и Adele с помощью метрики TTR (Type-Token Ratio). 
    1. Сначала вычисляет показатель TTR для каждой песни - это отношение количества уникальных слов к общему количеству слов в тексте, что показывает богатство словарного запаса (чем ближе к 1, тем разнообразнее лексика).
    2. Добавляет рассчитанные значения TTR в DataFrame для каждого исполнителя, предварительно обработав тексты (удалив стоп-слова и приведя слова к начальной форме).
    3. Создает визуализацию в виде двух боксплотов - для Billie Eilish и Adele отдельно, где по оси X указаны альбомы, а по оси Y - значения TTR, что позволяет сравнить разнообразие лексики между альбомами.
    4. Получившаяся сравнительная диаграмма сохранена в файл 'wordclouds/ttr_comparison.png'.
    5. В завершение выводит средние значения TTR по всем песням для каждого исполнителя.

In [23]:
# Функция для расчета TTR
def calculate_ttr(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

# Добавляем TTR в DataFrame
billie_df['ttr'] = billie_df['lyrics'].apply(lambda x: calculate_ttr(process_text(x)))
adele_df['ttr'] = adele_df['lyrics'].apply(lambda x: calculate_ttr(process_text(x)))

# Визуализация TTR по альбомам
plt.figure(figsize=(16, 8))

# Для Billie Eilish
plt.subplot(1, 2, 1)
sns.boxplot(data=billie_df, x='album_name', y='ttr', palette='Blues')
plt.title('Лексическое разнообразие (Billie Eilish)', fontsize=14)
plt.xlabel('Альбом', fontsize=12)
plt.ylabel('TTR (Type-Token Ratio)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Для Adele
plt.subplot(1, 2, 2)
sns.boxplot(data=adele_df, x='album_name', y='ttr', palette='Oranges')
plt.title('Лексическое разнообразие (Adele)', fontsize=14)
plt.xlabel('Альбом', fontsize=12)
plt.ylabel('')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('wordclouds/ttr_comparison.png', bbox_inches='tight', dpi=300)
plt.close()

# Сравнение средних значений TTR
print(f"Средний TTR Billie Eilish: {billie_df['ttr'].mean():.3f}")
print(f"Средний TTR Adele: {adele_df['ttr'].mean():.3f}")

Средний TTR Billie Eilish: 0.563
Средний TTR Adele: 0.494


**Вывод по результатам TTR**
1. Лексическое разнообразие
    - У Billie Eilish средний TTR (0.563) выше, чем у Adele (0.494), что означает, что её тексты более разнообразны в плане используемой лексики.  
2. Стилистические различия
   - Billie Eilish чаще использует разные слова в песнях, что может быть связано с её экспериментальным стилем, нестандартными темами и более разговорной манерой песен.  
   - Adele демонстрирует*меньший TTR, что типично для артистов с более традиционной поп- и соул-лирикой, где повторяются ключевые эмоционально заряженные слова (например, love, heart, baby).  

In [26]:
# 1. Распределение длины песен
def plot_song_length_distribution(df1, df2, name1, name2):
    
    """
    Сравнивает распределение длины песен между двумя исполнителями
    с помощью гистограмм и boxplot.
    """
    
    plt.figure(figsize=(18, 12))
    
    # Гистограммы для сравнения средней длины
    plt.subplot(2, 2, 1)
    sns.histplot(df1['word_count'], color='blue', alpha=0.5, label=name1, kde=True)
    sns.histplot(df2['word_count'], color='orange', alpha=0.5, label=name2, kde=True)
    plt.title('Распределение количества слов в песнях')
    plt.xlabel('Количество слов')
    plt.ylabel('Частота')
    plt.legend()
    
    plt.subplot(2, 2, 2)
    sns.histplot(df1['char_count'], color='blue', alpha=0.5, label=name1, kde=True)
    sns.histplot(df2['char_count'], color='orange', alpha=0.5, label=name2, kde=True)
    plt.title('Распределение количества символов в песнях')
    plt.xlabel('Количество символов')
    plt.ylabel('Частота')
    plt.legend()
    
    # Boxplot'ы для сравнения по альбомам
    plt.subplot(2, 2, 3)
    sns.boxplot(data=pd.concat([
        df1.assign(artist=name1), 
        df2.assign(artist=name2)
    ]), x='artist', y='word_count', palette=['blue', 'orange'])
    plt.title('Распределение слов по исполнителям')
    plt.xlabel('Исполнитель')
    plt.ylabel('Количество слов')
    
    plt.subplot(2, 2, 4)
    sns.boxplot(data=pd.concat([
        df1.assign(artist=name1), 
        df2.assign(artist=name2)
    ]), x='artist', y='char_count', palette=['blue', 'orange'])
    plt.title('Распределение символов по исполнителям')
    plt.xlabel('Исполнитель')
    plt.ylabel('Количество символов')
    
    plt.tight_layout()
    plt.savefig('visualizations/song_length_comparison.png', dpi=300)
    plt.close()

# 2. Сложность текста (индекс удобочитаемости Flesch-Kincaid)
def calculate_readability(text):
   
    """
    Вычисляет индекс удобочитаемости Flesch-Kincaid
    """
    
    try:
        sentences = nltk.sent_tokenize(text)
        words = nltk.word_tokenize(text)
        
        if len(sentences) == 0 or len(words) == 0:
            return 0
        
        # Количество слогов (упрощенный метод)
        syllable_count = sum([len(re.findall(r'[aeiouy]+', word.lower())) for word in words])
        
        # Формула Flesch-Kincaid
        readability = 206.835 - 1.015 * (len(words)/len(sentences)) - 84.6 * (syllable_count/len(words))
        return readability
    except:
        return 0  # Return 0 if there's any error in processing

def plot_readability_comparison(df1, df2, name1, name2):
   
    """
    Сравнивает удобочитаемость текстов между исполнителями
    """
    
    # Добавляем столбец с индексом удобочитаемости
    df1['readability'] = df1['lyrics'].apply(calculate_readability)
    df2['readability'] = df2['lyrics'].apply(calculate_readability)
    
    plt.figure(figsize=(12, 6))
    
    # Boxplot сравнения
    sns.boxplot(data=pd.concat([
        df1.assign(artist=name1), 
        df2.assign(artist=name2)
    ]), x='artist', y='readability', palette=['blue', 'orange'])
    
    plt.title('Сравнение индекса удобочитаемости Flesch-Kincaid')
    plt.xlabel('Исполнитель')
    plt.ylabel('Индекс удобочитаемости')
    plt.savefig('visualizations/readability_comparison.png', dpi=300)
    plt.close()
    
    # Выводим средние значения
    print(f"Средняя удобочитаемость {name1}: {df1['readability'].mean():.1f}")
    print(f"Средняя удобочитаемость {name2}: {df2['readability'].mean():.1f}")

# Выполняем все анализы
plot_song_length_distribution(billie_df, adele_df, 'Billie Eilish', 'Adele')
plot_readability_comparison(billie_df, adele_df, 'Billie Eilish', 'Adele')

Средняя удобочитаемость Billie Eilish: 0.0
Средняя удобочитаемость Adele: 0.0


**Вывод**
1. Читаемость 
    - Оценка Flesch-Kincaid не применима к текстам песен в классическом понимании, что отражается в нулевых значениях для обеих исполнительниц.
2. Длина песен 
    - Песни Adele в среднем содержат больше слов и символов, чем песни Billie Eilish, что говорит о большей длине и, возможно, детализации текстов Adele.
3. Структура текстов
    - Тексты обеих исполнительниц имеют особенности, типичные для песен - короткие строки, повторения, что влияет на показатели читаемости и длины.

In [34]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import json
import plotly.graph_objects as go
from collections import Counter
import nltk
from nltk.corpus import stopwords
import string
import re
import textstat  # для метрик читаемости

# Загрузка данных (Файлы уже есть в директории проекта, тамже где и ноутбук)
billie_df = pd.read_csv('billie_eilish_songs.csv')
adele_df = pd.read_csv('adele_songs.csv')

# Добавляем метрики для каждого текста
def calculate_text_metrics(df):
    df['ttr'] = df['lyrics'].apply(calculate_ttr)
    df['flesch_reading'] = df['lyrics'].apply(lambda x: textstat.flesch_reading_ease(str(x)))
    return df

def calculate_ttr(text):
    try:
        text = str(text).lower()
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        words = text.split()
        words = [word for word in words if len(word) > 2]
        if not words:
            return 0
        unique_words = set(words)
        return len(unique_words) / len(words)
    except:
        return 0

# Применяем расчет метрик
billie_df = calculate_text_metrics(billie_df)
adele_df = calculate_text_metrics(adele_df)

combined_df = pd.concat([billie_df, adele_df])

# Инициализация Dash app
app = dash.Dash(__name__)

# Улучшенная функция для создания данных для облака слов
def generate_wordcloud_data(text, max_words=100):
    try:
        # Очистка текста
        text = text.lower()
        text = re.sub(r'\[.*?\]', '', text)  # Удаляем текст в квадратных скобках
        text = re.sub(r'[^\w\s]', '', text)  # Удаляем пунктуацию
        words = text.split()  # Простое разделение по пробелам вместо word_tokenize
        
        # Удаляем стоп-слова и короткие слова
        stop_words = set(stopwords.words('english'))
        words = [word for word in words if word not in stop_words and len(word) > 2]
        
        word_freq = Counter(words)
        return word_freq.most_common(max_words)
    except Exception as e:
        print(f"Error in generate_wordcloud_data: {e}")
        return []

# Создание layout
app.layout = html.Div([
    html.H1("Анализ текстов песен Billie Eilish и Adele", style={'textAlign': 'center'}),
    
    dcc.Tabs([
        dcc.Tab(label='Общая статистика', children=[
            html.Div([
                dcc.Dropdown(
                    id='artist-selector',
                    options=[
                        {'label': 'Billie Eilish', 'value': 'Billie Eilish'},
                        {'label': 'Adele', 'value': 'Adele'},
                        {'label': 'Оба исполнителя', 'value': 'Both'}
                    ],
                    value='Both',
                    style={'width': '50%', 'margin': '0 auto'}
                ),
                dcc.Graph(id='word-count-plot'),
                dcc.Graph(id='char-count-plot'),
                dcc.Graph(id='album-stats-plot')
            ])
        ]),
        
        dcc.Tab(label='Анализ слов', children=[
            html.Div([
                dcc.Dropdown(
                    id='wordcloud-artist',
                    options=[
                        {'label': 'Billie Eilish', 'value': 'Billie Eilish'},
                        {'label': 'Adele', 'value': 'Adele'}
                    ],
                    value='Billie Eilish',
                    style={'width': '50%', 'margin': '0 auto'}
                ),
                dcc.Graph(id='wordcloud-plot'),
                dcc.Graph(id='top-words-plot')
            ])
        ]),
        
        dcc.Tab(label='Сравнение', children=[
            html.Div([
                dcc.Graph(id='ttr-comparison'),
                dcc.Graph(id='readability-comparison'),
                dcc.Graph(id='word-count-comparison')
            ])
        ])
    ])
])

# Callbacks для интерактивности
@app.callback(
    [Output('word-count-plot', 'figure'),
     Output('char-count-plot', 'figure'),
     Output('album-stats-plot', 'figure')],
    [Input('artist-selector', 'value')]
)
def update_stats(artist):
    if artist == 'Both':
        df = combined_df
        title_suffix = 'Оба исполнителя'
    else:
        df = combined_df[combined_df['artist'] == artist]
        title_suffix = artist
    
    # График количества слов
    word_fig = px.box(df, x='artist', y='word_count', 
                     title=f'Распределение количества слов ({title_suffix})',
                     color='artist')
    
    # График количества символов
    char_fig = px.box(df, x='artist', y='char_count', 
                     title=f'Распределение количества символов ({title_suffix})',
                     color='artist')
    
    # График статистики по альбомам
    album_counts = df.groupby(['artist', 'album_name']).size().reset_index(name='count')
    album_fig = px.bar(album_counts,
                      x='album_name', y='count', 
                      title=f'Количество песен по альбомам ({title_suffix})',
                      color='artist', barmode='group')
    
    return word_fig, char_fig, album_fig

@app.callback(
    [Output('wordcloud-plot', 'figure'),
     Output('top-words-plot', 'figure')],
    [Input('wordcloud-artist', 'value')]
)
def update_word_analysis(artist):
    if artist == 'Billie Eilish':
        text = ' '.join(billie_df['lyrics'].astype(str))
    else:
        text = ' '.join(adele_df['lyrics'].astype(str))
    
    # Облако слов
    wordcloud_data = generate_wordcloud_data(text)
    wordcloud_df = pd.DataFrame(wordcloud_data, columns=['word', 'count'])
    wordcloud_fig = px.bar(wordcloud_df,
                         x='word', y='count',
                         title=f'Частота слов ({artist})')
    
    # Топ 20 слов
    top_words = wordcloud_data[:20]
    top_words_df = pd.DataFrame(top_words, columns=['word', 'count'])
    top_words_fig = px.bar(top_words_df,
                         x='count', y='word', orientation='h',
                         title=f'Топ 20 слов ({artist})')
    
    return wordcloud_fig, top_words_fig

@app.callback(
    [Output('ttr-comparison', 'figure'),
     Output('readability-comparison', 'figure'),
     Output('word-count-comparison', 'figure')],
    [Input('artist-selector', 'value')]  # Это input не используется, но нужен для callback
)
def update_comparison(_):
    # Сравнение TTR (Type-Token Ratio)
    ttr_fig = px.box(combined_df, x='artist', y='ttr', 
                    title='Сравнение TTR (Type-Token Ratio)',
                    color='artist')
    
    # Сравнение читаемости
    readability_fig = px.box(combined_df, x='artist', y='flesch_reading', 
                           title='Сравнение уровня читаемости (Flesch Reading Ease)',
                           color='artist')
    
    # Сравнение количества слов
    word_count_fig = px.box(combined_df, x='artist', y='word_count',
                           title='Сравнение количества слов в песнях',
                           color='artist')
    
    return ttr_fig, readability_fig, word_count_fig

# Запуск приложения
if __name__ == '__main__':
    app.run(debug=True, port=8051)
